# Multi-Injection Pipeline with Rubin PSF

This notebook demonstrates injecting clusters using the Rubin CoaddPsf, 
performing injections in batches, and combining results for completeness analysis.

In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
from src.inject import inject_clusters_in_batches, create_injection_catalog

# Set up paths and configurations
output_dir = 'outputs/multi_injection_rubin_psf'
os.makedirs(output_dir, exist_ok=True)

# Rubin data configuration
butler = Butler('dp02', collections='2.2i/runs/DP0.2')
data_id = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd = butler.get('deepCoadd', dataId=data_id)
image = coadd.image.array.copy()
psf_obj = coadd.getPsf()
bbox = coadd.getBBox()
BBOX_X_MIN, BBOX_Y_MIN = bbox.getMinX(), bbox.getMinY()

print(f'Loaded deepCoadd: tract={data_id["tract"]}, patch={data_id["patch"]}, band={data_id["band"]}')

In [ ]:
# Generate injection catalog
n_clusters = 10000
batch_size = 1000
catalog = create_injection_catalog(
    n_clusters=n_clusters,
    image_shape=image.shape,
    mag_range=(20, 26),
    r_half_range=(2, 10),
    profile_type='king',
    edge_buffer=50,
    exposure=coadd,
    seed=42
)
print(f'Generated catalog with {len(catalog)} clusters.')

In [ ]:
# Perform injections in batches
injected_image, injection_info = inject_clusters_in_batches(
    image=image,
    catalog=catalog,
    batch_size=batch_size,
    exposure=coadd,
    add_noise=True,
    pixel_scale=0.2,
    save_stamps=False,
    psf_mode='spatially_varying'
)

# Save the injected image
np.save(os.path.join(output_dir, 'injected_image.npy'), injected_image)
print(f'Injected image saved to {output_dir}/injected_image.npy')

In [ ]:
# Placeholder for user detection pipeline
detections = []  # Replace with actual detection results
print('Run your detection pipeline and populate the `detections` list.')

In [ ]:
# Combine detection results and analyze completeness
from src.retrieval import ClusterRetrieval

if detections:
    retrieval = ClusterRetrieval(injection_info, detections)
    retrieval.match_detections(match_radius=5.0)
    stats = retrieval.get_summary_statistics()

    print('=== Completeness Results ===')
    print(f'Injected clusters: {stats["n_injected"]}')
    print(f'Recovered clusters: {stats["n_detected"]}')
    print(f'Completeness: {stats["overall_completeness"]:.1%}')
else:
    print('No detections provided. Completeness analysis skipped.')